In [11]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent
RESULTS = ROOT/"results"

# Path has its own glob, glob.glob() only takes a string
json_files = sorted(RESULTS.glob("*_summary.json"))
# skip the 10-case smoke run
json_files = [f for f in json_files if "smoke" not in f.name]

# each file is one flat dict, so read it as a Series (one row)
rows = [pd.read_json(f, typ="series") for f in json_files]
final_df = pd.DataFrame(rows)
final_df["margin"] = 1.96 * ((final_df["accuracy"] * (1 - final_df["accuracy"])) / final_df["cases"]) ** 0.5

# add tokens/sec from benchmark.py (results/<variant>_benchmark.json); variants without one stay empty
bench_files = sorted(RESULTS.glob("*_benchmark.json"))
bench_df = pd.DataFrame([pd.read_json(f, typ="series") for f in bench_files])
final_df = final_df.merge(bench_df[["variant", "generation_tps", "prompt_tps"]], on="variant", how="left")
print(final_df)

     variant  cases  accuracy  seconds_per_case  peak_memory_gb  accuracy_bug  \
0  4bit_lora    300     0.817             0.422            2.93          0.80   
1       4bit    300     0.670             0.314            2.51          0.92   
2       8bit    300     0.663             0.418            3.96          0.83   
3       bf16    300     0.663             0.513            6.90          0.83   

   accuracy_feature  accuracy_question    margin  generation_tps  prompt_tps  
0              0.84               0.81  0.043755             NaN         NaN  
1              0.81               0.28  0.053210            61.7      1438.0  
2              0.87               0.29  0.053489            34.7      1258.3  
3              0.86               0.30  0.053489            17.5       835.5  


In [14]:
from math import comb

a = pd.read_csv(RESULTS/"4bit_lora.csv").set_index("id")["correct"]
b = pd.read_csv(RESULTS/"4bit_lora_fused.csv").set_index("id")["correct"]
b = b.loc[a.index]   # same issue order as a

only_a = int((a & ~b).sum())   # adapter right, fused wrong
only_b = int((~a & b).sum())   # fused right, adapter wrong
n = only_a + only_b
k = min(only_a, only_b)
p = min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2 ** n)
print(only_a, only_b, p)


54 19 5.0622659111780655e-05


In [15]:
final_df.round(3).to_csv(RESULTS/"comparison.csv", index=False)